# 🧠 EX64: Callback Hooks & Custom Metric Logging

YOLO fires Python callables at lifecycle events — you add logic without modifying YOLO internals.

| Hook | Fires when |
|------|-----------|
| `on_train_start` | Before first epoch |
| `on_train_epoch_end` | After train loss, **before** val |
| `on_fit_epoch_end` | After **both** train + val |
| `on_val_end` | After standalone `model.val()` |
| `on_train_end` | After all epochs |

**Key:** Use `on_fit_epoch_end` (not `on_train_epoch_end`) to read `metrics/mAP50(B)` —
that key is only available **after** validation runs.

## 🔗 Links
- [[EX53_Training_Settings_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import register_custom_metric_callback
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

print("\n--- AUDIT & INSPECTION START ---")
model = YOLO("yolo11n.pt")
print(f"Pre-reg 'on_val_end': {model.callbacks.get('on_val_end', [])}")

model = register_custom_metric_callback(model)
post_cbs = model.callbacks.get("on_val_end", [])
print(f"Post-reg 'on_val_end': {[cb.__name__ for cb in post_cbs]}")
print(f"Total hooks across all events: {sum(len(v) for v in model.callbacks.values())}")

print("\nRunning model.val() on coco8 to trigger callback...")
try:
    results = model.val(data="coco8.yaml", imgsz=320, device=device, verbose=False)
    mAP50 = results.box.map50
    mAP95 = results.box.map
    mp = results.box.mp
    mr = results.box.mr
    print(f"  Precision={mp:.4f}  Recall={mr:.4f}")
    print(f"  mAP@0.5={mAP50:.4f}  mAP@0.5:0.95={mAP95:.4f}")
except Exception as e:
    print(f"Validation error: {e}")
    mAP50=mAP95=mp=mr=0.0
print("--- AUDIT & INSPECTION END ---")

fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].bar(["mAP@0.5","mAP@0.5:0.95"], [mAP50,mAP95], color=["#27ae60","#145a32"], edgecolor="black")
axes[0].set_ylim(0,1.0); axes[0].set_ylabel("Score"); axes[0].set_title("mAP (via callback)")
for i,v in enumerate([mAP50,mAP95]): axes[0].text(i, v+0.02, f"{v:.3f}", ha="center", fontweight="bold")
axes[1].bar(["Precision","Recall"], [mp,mr], color=["#2980b9","#8e44ad"], edgecolor="black")
axes[1].set_ylim(0,1.0); axes[1].set_ylabel("Score"); axes[1].set_title("P & R")
for i,v in enumerate([mp,mr]): axes[1].text(i, v+0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
